# AgentState 状态管理

Agent 在执行过程中需要维护状态——消息历史、结构化响应、流程控制等。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## AgentState 结构
AgentState 是一个 TypedDict，默认包含三个字段：

In [4]:
from typing import Annotated
from typing_extensions import Required, NotRequired
from langgraph.graph.message import add_messages
from langgraph.channels.ephemeral_value import EphemeralValue
from langchain.messages import AnyMessage
from typing import Any
from typing_extensions import TypedDict


# AgentState 的实际定义（简化版）
class AgentState(TypedDict):
    # messages：消息历史，使用 add_messages 作为 reducer
    # Required 表示调用时必须提供
    messages: Required[Annotated[list[AnyMessage], add_messages]]

    # jump_to：流程跳转控制，ephemeral（使用后自动清除）
    # NotRequired 表示可选
    jump_to: NotRequired[Annotated[str | None, EphemeralValue]]

    # structured_response：结构化输出结果
    # NotRequired 表示可选，仅在 response_format 设置时出现
    structured_response: NotRequired[Any]

| 字段 | 类型 | 必填 | 说明 |
| --- | --- | --- | --- |
| messages | list[AnyMessage] | 是 | 消息历史，add_messages reducer |
| jump_to | str | 否 | 流程跳转（tools/model/end），ephemeral |
| structured_response | Any | 否 | 结构化输出结果 |

## messages——消息历史的 Reducer 机制

messages 字段使用 `add_messages` reducer，更新时追加而非覆盖。

In [5]:
from langchain.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages

messages = [
    HumanMessage(content="你好", id="1"),
    AIMessage(content="你好！", id="2"),
]
new_msg = AIMessage(content="有什么可以帮你的？", id="3")
result = add_messages(messages, [new_msg])

print(f"合并前: {len(messages)} 条")
print(f"合并后: {len(result)} 条")
for msg in result:
    print(f"  [{msg.type}] {msg.content}")

合并前: 2 条
合并后: 3 条
  [human] 你好
  [ai] 你好！
  [ai] 有什么可以帮你的？


`add_messages` 智能特性：同名覆盖（ID 相同替换）、支持 RemoveMessage、类型安全。

## jump_to——流程跳转控制

Middleware 中最常用的字段，ephemeral 瞬态，使用后自动清除。

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

@before_model
def check_question(state, runtime):
    """检查问题是否合法"""
    messages = state.get("messages", [])
    if not messages:
        return None
    if "密码" in str(messages[-1].content):
        return {"jump_to": "end", "messages": [HumanMessage(content="抱歉，不能回答关于密码的问题。")]}
    return None

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, middleware=[check_question], system_prompt="你是菜鸟教程 RUNOOB 的助手。")

print("jump_to 中间件已注册")

jump_to 中间件已注册


| jump_to 值 | 跳转到 | 效果 |
| --- | --- | --- |
| "tools" | 工具执行节点 | 跳过模型调用 |
| "model" | 模型节点 | 让模型重新处理 |
| "end" | 结束 Agent 循环 | 跳转到结束 |

## structured_response——结构化输出

使用 `response_format` 参数时，结果存储在此字段。

In [4]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

class CourseRecommendation(BaseModel):
    course_name: str = Field(description="推荐课程")
    reason: str = Field(description="推荐理由")
    difficulty: str = Field(description="难度")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, response_format=CourseRecommendation, system_prompt="你是菜鸟教程 RUNOOB 的学习顾问。")

print("结构化输出 Agent 已创建")

结构化输出 Agent 已创建


## 自定义 State 扩展

继承 `AgentState` 添加业务字段。

In [6]:
from typing import Annotated
from langchain.agents import create_agent, AgentState
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool, InjectedState
from typing_extensions import TypedDict


# 扩展 AgentState，添加业务字段
class ShoppingAgentState(AgentState):
    """购物助手的状态"""
    cart: list[str]         # 购物车商品列表
    total_price: float      # 总价


@tool
def add_to_cart(
    item: str,
    price: float,
    state: Annotated[dict, InjectedState],
) -> str:
    """将商品添加到购物车。

    Args:
        item: 商品名称
        price: 商品价格
    """
    cart = state.get("cart", [])
    total = state.get("total_price", 0.0)

    return {
        "cart": cart + [item],
        "total_price": total + price,
        "messages": [],  # 不添加额外消息
    }


@tool
def view_cart(
    state: Annotated[dict, InjectedState],
) -> str:
    """查看购物车内容"""
    cart = state.get("cart", [])
    total = state.get("total_price", 0.0)
    if not cart:
        return "购物车为空"
    items = "、".join(cart)
    return f"购物车：{items}，总价：¥{total:.2f}"


model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model,
    tools=[add_to_cart, view_cart],
    state_schema=ShoppingAgentState,  # 使用自定义状态
    system_prompt="你是菜鸟教程 RUNOOB 商店的购物助手。",
)

# 初始状态包含空的购物车
result = agent.invoke({
    "messages": [HumanMessage(content="帮我加一本 Python 教程到购物车，价格 49.9")],
    "cart": [],
    "total_price": 0.0,
})

print(f"购物车: {result.get('cart', [])}")
print(f"总价: ¥{result.get('total_price', 0):.2f}")
print(f"回复: {result['messages'][-1].content}")

购物车: []
总价: ¥0.00
回复: 已经成功将 **Python 教程** 添加到购物车啦！🛒

当前购物车信息如下：
- **商品**：Python 教程
- **价格**：¥49.9
- **购物车总价**：¥49.9

请问还需要添加其他商品吗？😊


## state_schema vs middleware state_schema
既可以通过 create_agent() 的 state_schema 参数扩展状态，也可以通过 Middleware 的 state_schema 扩展。两者的区别：
| 方式	| 使用场景 | 优先级 |
| ---| --- | --- |
| create_agent(state_schema=...) | 全局状态扩展，所有节点共享 | 最高（覆盖 middleware 同名字段） |
| AgentMiddleware(state_schema=...) | 特定 middleware 的状态扩展 | 较低，可被 create_agent 覆盖 |

**推荐做法**：将通用的业务状态字段放在 state_schema 中，将特定 middleware 相关的内部字段放在 middleware 的 state_schema 中。这样职责清晰，不会相互污染。